# Bayesian-DGCRAN — Kaggle GPU Experiment Runner (All 6 Datasets)
1. Attach your Kaggle Private Dataset (`pems-metr-traffic-benchmarks`) containing `PeMS03`, `PeMS04`, `PeMS07`, `PeMS08`, `METR-LA`, `PEMS-BAY`.
2. Set **Accelerator = GPU P100 (or T4 x2)** and **Persistence = Files only**.
3. Choose `DATASET` below and click **Save & Run All (Commit)** for unattended 12-hour execution.

In [ ]:
import os, torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Clone from your GitHub repo (or unzip bayesian-dgcran-thesis-phase2-complete.zip)
# !git clone https://github.com/<YOUR_USERNAME>/bayesian-dgcran-thesis.git
# %cd bayesian-dgcran-thesis

DATA_ROOT = '/kaggle/input/pems-metr-traffic-benchmarks'
!python3 scripts/prepare_datasets.py {DATA_ROOT}
!python3 -m unittest discover -s tests -v
!python3 scripts/smoke_test_gpu.py

In [ ]:
DATASET = 'PeMS08'  # Run one per Kaggle session: PeMS08, PeMS04, PeMS03, METR-LA, PEMS-BAY, PeMS07
OUT_DIR = f'/kaggle/working/results_{DATASET}'

MODELS = [
    'HA', 'VAR', 'DCRNN', 'AGCRN', 'DGCRN', 'DGCRAN',
    'MCDropout-DGCRAN', 'DeepSTUQ', 'DeepEnsembles',
    'Bayesian-DGCRAN',
    'Ablation-NoVNAPL', 'Ablation-NoBDAGM', 'Ablation-NoVarMasks',
    'Ablation-NoKLAWA', 'Ablation-NoNEMHCC'
]

for m in MODELS:
    print(f'\n================ Running {m} on {DATASET} ================')
    !python3 scripts/run_experiment.py --dataset {DATASET} --model {m} --data_root {DATA_ROOT} --output_dir {OUT_DIR} --epochs 80 --warmup_epochs 20 --awa_epochs 20 --seed 42

In [ ]:
# Robustness Study (10% to 50% Missing Sensor Data on Bayesian-DGCRAN)
for miss in [0.1, 0.2, 0.3, 0.4, 0.5]:
    !python3 scripts/run_experiment.py --dataset {DATASET} --model Bayesian-DGCRAN --data_root {DATA_ROOT} --output_dir {OUT_DIR} --missing_rate {miss} --epochs 80 --seed 42